In [0]:
%pip install dotenv
%pip install langchain_community
%pip install langchain_huggingface
%pip install langchain-openai==0.3.21
%pip install pymupdf
%pip install git+https://github.com/huggingface/transformers.git
#%env PIP_DISABLE_PIP_VERSION_CHECK=1%pip install faiss-gpu
%pip install faiss-cpu
%pip install rank_bm25

In [0]:
%restart_python

In [0]:
import os
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
from langchain_community.vectorstores import FAISS
from typing import List
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyMuPDFLoader

In [0]:
def find_pdfs(root_dir: str) -> List[Path]:
    root = Path(root_dir)
    # The rglob pattern is case-sensitive by default; to catch .PDF too, you could check suffix lowercased:
    pdfs = [p for p in root.rglob('*') if p.is_file() and p.suffix.lower() == '.pdf']
    return pdfs


t = []
files = find_pdfs('./data/')

for i in files:
    FILE_PATH = str(i)  

    doc = PyMuPDFLoader(FILE_PATH, mode='page') # open a document
    text = doc.load()

    for i in range(len(text)):
        text[i].metadata['page'] += 1
    t.extend(text)

load_dotenv()

print(len(t))

text_splitter = RecursiveCharacterTextSplitter(
    # Set a really small chunk size, just to show.
    chunk_size=1000,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)

texts = text_splitter.split_documents(t)

try:
    os.environ['OPENAI_API_KEY'] = dbutils.secrets.get(scope='common', key="NetDoc_OPENAI_KEY")
except NameError:
    raise EnvironmentError("dbutils is not available. Ensure you're running this in a supported environment or provide the OpenAI API key manually.")

TOP_K = 5

In [0]:
EMBED_MODEL_ID = "Qwen/Qwen3-Embedding-0.6B"

embedding = HuggingFaceEmbeddings(model_name=EMBED_MODEL_ID, model_kwargs={'device': 'cuda'})

vector_store = FAISS.from_documents(texts, embedding)

In [0]:
vector_store.save_local("faiss_index")

In [0]:
vector_store1 = FAISS.load_local("faiss_index", embedding, allow_dangerous_deserialization=True)

In [0]:
retriever = vector_store1.as_retriever(search_kwargs={"k": TOP_K})

In [0]:
keyword_retriever = BM25Retriever.from_documents(t)
keyword_retriever.k =  5
ensemble_retriever = EnsembleRetriever(retrievers=[retriever,keyword_retriever],weights=[0.5, 0.5])

In [0]:
user_prompt = "What do you do when you get no response from a get request?"

retr = ensemble_retriever.get_relevant_documents(user_prompt)

texts= [(d.page_content, d.metadata) for d in retr]

print(texts[0])

In [0]:
#Set up filter of metadata here

def slim_metadata(documents):
    """
    documents: list of tuples (text, metadata_dict)
    returns: new list of tuples (text, {'source': ..., 'page': ...})
    """
    slimmed = []
    for text, meta in documents:
        # pull only the two keys you care about
        src  = meta.get('file_path')  # or meta.get('source') if you renamed it earlier
        page = meta.get('page')
        slimmed.append((text, {'source': src, 'page': page}))
    return slimmed

slimmed_docs = [
    (text, {'source': m['file_path'], 'page': m['page']})
    for text, m in texts
]

print(slimmed_docs)


In [0]:
llm = ChatOpenAI(model='gpt-4o-mini')

prompt = PromptTemplate.from_template("You are currently a part of a RAG. Using these files: {input}, answer the user's prompt: {prompt} with the at most precesion possible. The files you have received are always organized with the page content first and then then metadata, so be sure to associate the right documents with the right metadata. Keep your answers clear and with as little filler as possible. Be to the point! If you believe that the piece of information within the prompt is not found in the given document chunks, say so rather than making up information. At the begining of your answer, cite the prompt given to you. At the end of your answer, give the source of the document your found the information in as well as the page number. If you found information from multiple pages or multpible document, cite them all.")

chain = prompt | llm
print(chain.invoke(
    {
        "prompt": user_prompt,
        "input": slimmed_docs
    }
).content)